# README

- **Author**: `李芸榛（JenniferjjLee）`
- **Created At**: `2026-11-08`
- **Last Modified At**: `2026-11-08`（v7：修正截斷站名「捷運科技大樓站(台北教育大學」；同日改用新版 `stations_in_field`，臺大人文館改用正式 StationID，並納入羅斯福路三段283巷）

---

## What does this file do?

- **研究範圍 = 新版 `stations_in_field`（2026-11-08 版，109 站，全部有 StationID）**，從 24 個原始月檔重新篩選。
  - 新版清單比 2026-11-07 版多兩個 2025 年才啟用的站：`臺大人文館`（`500119097`，第一筆交易 2025-02-04）和 `羅斯福路三段283巷`（`500101265`，2025-10-10 起）。
  - 先前臺大人文館因舊清單沒有 ID，暫用 `-2`；本版改用正式 StationID，`-2` 不再出現。羅斯福路三段283巷先前算範圍外（`-1`），依組員決定（2026-11-08）本版納入範圍。
- **v7 修正截斷站名**（2026-11-08）：原始資料 2024-01～2025-01 把 `捷運科技大樓站(台北教育大學)`（StationID `500101166`）寫成少了右括號的 `捷運科技大樓站(台北教育大學`；2025-02 起才全部是完整站名。v6 用精確比對，截斷站名被當成範圍外（`-1`），兩端都不在範圍內的交易則整筆漏掉。v7 在篩選前先依 `NAME_ALIASES` 把截斷站名改成完整站名，再照原規則篩選與編碼。這是唯一的站名修正；其他站名仍不修改。
- 觀測單位是一筆來源交易紀錄。借車站或還車站任一端與 109 個站名之一（去掉 `YouBike2.0_` 前綴）**精確**相同即保留（OR）；兩端都相同也只輸出一次。不去重、不補值，除 `NAME_ALIASES` 以外不修正站名；年度依來源檔月份決定，其餘規則與 v3–v6 相同。
- 讀檔沿用 `YouBike_traffic_raw.py`（stage2）已驗證的 `chunks()` 與 `serialize_selected()`；亂碼 bytes 以 `<UNRESOLVED_UTF8_HEX:...>` 表示。
- **站名**：所有非空站名都加前綴 `YouBike2.0_`；空白站名與亂碼 token 保持原樣。
- **StationID**（`rent_station_id`、`return_station_id`）：
  - 109 站：`stations_in_field` 的 9 碼 StationID（4 個同名移位站合併成一對一對照）。
  - 其他站、空白站名、亂碼 token：`-1`（範圍外，不是真實站點代碼）。
- **與 v6 的關係**：v7 不讀 v6，可以獨立重跑。v6 檔案保留在原處供比對；與 v6 的逐列比對在本 notebook 之外另做一次（見結果摘要）。
- **分組欄 `trip_group`**（2026-11-08 加入，放在最後一欄；依組員定義，四組互斥、先判斷 d）：
  - `a` 外借台大還（流入）：借車站 ID = `-1`，還車站 ID ≠ `-1`。
  - `b` 台大借外還（流出）：借車站 ID ≠ `-1`，還車站 ID = `-1`。
  - `c` 台大借台大還（區內）：兩端 ID 都 ≠ `-1`，且不屬於 d。
  - `d` 無效還車：借還同站，且 `rent` 時長 ≤ 180 秒（3 分鐘）。
  - 「台大」= 研究範圍 109 站。
- 範例檔取 2025 v7 依檔案順序的前 100 列（2026-11-08 由 2024 改為 2025）。

---

## What does this file take?

- **Source Data Sets** (one entry per actual file):
  1. `data/raw/2024-25YouBike2.0票證刷卡資料/raw_data/YYYYMM_YouBike2.0票證刷卡資料.csv`（202401–202512，共 24 個檔）
     - Input key: `raw_YYYYMM`
     - Description: 臺北市 YouBike 2.0 票證刷卡原始資料，合計 133,966,680 筆。
     - Documentation: `data/raw/YouBike_traffic_codebook.md`
     - SHA-256: 每個月檔各自寫在下方 `EXPECTED_RAW_SHA256`，每月筆數寫在 `EXPECTED_RAW_ROWS`（取自第一階段稽核 `stage1_res/`）。
  2. `data/processed/stations_in_field.csv`
     - Input key: `stations_in_field`
     - Description: 研究範圍內站點清單（2026-11-08 版，commit `0d658ca`），113 列 × 9 欄、109 站；使用 `StationNameZh_tw` 與 `StationID`。
     - Documentation: pending；產生程式 `data/code/filter_stations_in_field.R`
     - SHA-256: `74dadd7afe58f0b69b0a511f234dc85388e64290d5db420b8426767bad637f9e`

---

## What does this file output?

- `data/processed/youbike_traffic/YouBike_traffic_2024_v7.csv`
  - Output key: `traffic_2024`
  - Description: 2024（來源月份 202401–202412）借車或還車任一端在 109 個站名中的交易；站名加前綴，附 `rent_station_id`、`return_station_id`。
  - Role: `analysis`
  - Documentation: `data/processed/youbike_traffic/YouBike_traffic_doc.ipynb`
  - Expected schema and row-count rules: 10 欄 `rent_time, rent_station, rent_station_id, return_time, return_station, return_station_id, rent, bike_type, infodate, trip_group`；每列至少一端在清單內；109 站 ID 與清單一致、其餘為 `-1`；`trip_group` 符合分組規則；2024 年 107 站都要出現（兩個 2025 新站不會出現），2025 年 109 站都要出現；UTF-8-SIG。
  - Verification status / SHA-256: 見下方執行輸出與 `PROJECT-PROPOSAL.md` Data Artifacts
  - Downstream finding input: `data/processed/youbike_traffic/YouBike_traffic_2024_v7.csv` (same exact file)
- `data/processed/youbike_traffic/YouBike_traffic_2025_v7.csv`
  - Output key: `traffic_2025`
  - Description: 2025（來源月份 202501–202512），規則同上。
  - Role: `analysis`
  - Documentation: `data/processed/youbike_traffic/YouBike_traffic_doc.ipynb`
  - Expected schema and row-count rules: 同 2024。
  - Verification status / SHA-256: 見下方執行輸出與 `PROJECT-PROPOSAL.md` Data Artifacts
  - Downstream finding input: `data/processed/youbike_traffic/YouBike_traffic_2025_v7.csv` (same exact file)
- `data/processed/youbike_traffic/YouBike_traffic_processed_example_v7.csv`
  - Output key: `traffic_example`
  - Description: 2025 v7 依檔案順序的前 100 列（非抽樣、非排序），10 欄。
  - Role: 範例（檢視格式用，不作分析樣本）
  - Documentation: 本 Notebook
  - Expected schema and row-count rules: 100 列 × 10 欄，逐格等於 2025 v7 前 100 列。
  - Verification status / SHA-256: 見下方執行輸出與 `PROJECT-PROPOSAL.md` Data Artifacts
  - Downstream finding input: 不建議作分析輸入

All paths are relative to the project root, not the operating-system root.
兩個完整年度檔超過 GitHub 單檔 100 MB 上限，已列入 `data/processed/youbike_traffic/.gitignore`，需透過 repo 以外的管道分享；範例檔可以 commit。


In [1]:
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd
from IPython.display import display


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "AGENTS.md").is_file() and (candidate / "templates").is_dir():
            return candidate
    raise FileNotFoundError("Open this notebook within the student project containing AGENTS.md and templates/.")


PROJECT_ROOT = find_project_root()


def project_file(relative_path, area, must_exist=True):
    # Explicit project-relative paths only; never select the first matching file.
    rel = Path(relative_path)
    if rel.is_absolute() or ".." in rel.parts or "<" in str(rel):
        raise ValueError(f"Replace the placeholder with an exact project-relative path: {relative_path}")
    path = (PROJECT_ROOT / rel).resolve()
    allowed = (PROJECT_ROOT / "data" / area).resolve()
    if allowed == PROJECT_ROOT or not allowed.is_relative_to(PROJECT_ROOT):
        raise ValueError(f"data/{area}/ resolves outside the project root.")
    if not path.is_relative_to(allowed):
        raise ValueError(f"Expected a file inside data/{area}/: {relative_path}")
    if must_exist and not path.is_file():
        raise FileNotFoundError(path)
    return path


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def verify_input_sha256(path, expected):
    # Bind this run to the exact file inspected when the notebook was prepared.
    if (not isinstance(expected, str) or len(expected) != 64
            or any(char not in "0123456789abcdefABCDEF" for char in expected)):
        raise ValueError("Fill the expected SHA-256 from the inspected input before loading data.")
    actual = sha256_file(path)
    if actual != expected.lower():
        raise ValueError(f"Input changed: identify the new version before loading {path.name}.")
    return actual

In [2]:
import sys
from collections import Counter

BASE_DIR = "data/raw/2024-25YouBike2.0票證刷卡資料"
MONTHS = [f"{year}{month:02}" for year in (2024, 2025) for month in range(1, 13)]
RAW_INPUTS = {month: f"{BASE_DIR}/raw_data/{month}_YouBike2.0票證刷卡資料.csv" for month in MONTHS}
# Fingerprints and row counts of the 24 raw files, as verified by the stage-1 audit
# (stage1_res/input_sha256_before.json and stage1_repair_summary.csv).
EXPECTED_RAW_SHA256 = {
    "202401": "c31144a247f56d4364eb1412d346e6555f3956c83c8193ab200222ee7eb0d71e",
    "202402": "9e8f65f61b5343e5eeb06e9fb9c3603ab2d26a4293c275286aa495b87e95c0fd",
    "202403": "f799f5184ffbeee69cac2f688ce53912ebdba8d904ce8848e17d1b7498471a1f",
    "202404": "70962342ff2a085bf250877d80ddcfa283d9e344552e82a8b46edbaf27e71410",
    "202405": "7e91e909b31efc7c0c4948d904964df1a92b4e8a2b1c3cb26525e30d0a18129c",
    "202406": "dd23954ec298f98203488d278f72e40baf2e6a57539b34e863dc70dfa4471c0d",
    "202407": "12a16422f71a24fd2566fcafcd2e2aef9e32f5f06bf4a637adca55bdcbee7d23",
    "202408": "53130e05fe3988999a68e4dd6ea01784a16e6deb8dec4bcafcc51c506cc748f4",
    "202409": "079fa0a3138a9eadcc32724950f0359b58ac96117868d96f9a424e4e54547706",
    "202410": "00b80c52412b1f023f0f0e04201f2a546c270bde1d6c8846e29287826e3346fc",
    "202411": "fd6ef55345a769bf06fef2f7ff1213028e333468fb8cdf4c32bfb35c7a76b5b1",
    "202412": "0d7de25e428a49036dbbb9dcdf94841dc8ad21623de6a6280e7cb47c77bd4617",
    "202501": "712fba59b84530d6798f8b3457d1fe0f96458956377e2977750f2caef1202a20",
    "202502": "da1618f7922e96b847662895c84be8ad7caf5adaeba89def898c20eca2564114",
    "202503": "e8e355e74dc48cd1e977105bb3505b41812f6d9d52e98aeab0c5549eb67d163a",
    "202504": "f283e3783cb2ca83479c763639e7e89d98d84523f156e395cc5394ce14f25770",
    "202505": "5454436cebe2554148a9af2e054a97f5758692e7e45788a66be693303cde77ba",
    "202506": "43b401d4fa3579335ccd0a467b12ead36d6702ca64e993880b0d9f12b66d0aad",
    "202507": "669781855d96574e1db4f86166990ff77576bc7ba6bc00c211c37e70ac34b217",
    "202508": "23795936c673f7bb7f4f930af6485260707ba7a4520b79fbcaf319cac891ac34",
    "202509": "22791cb3ab8e1c84ce8026a1afb84ee3e9e37ace84f26926dec412334989b38a",
    "202510": "09989ebb841075c6ddbe172bd1b4f02908d84055cf08b56524762307edc21c62",
    "202511": "225636746356f42a58bb85bb9914d711a4d0e3f94acadca46eacff57e692bf85",
    "202512": "904d1279d9491f9f70ce0b5e0dcb800a1cfa3cf367dd5ed1238a6d88765f0eab",
}
EXPECTED_RAW_ROWS = {
    "202401": 2_746_557,
    "202402": 2_590_755,
    "202403": 3_472_416,
    "202404": 3_798_550,
    "202405": 4_338_453,
    "202406": 3_773_353,
    "202407": 5_528_656,
    "202408": 6_180_179,
    "202409": 5_799_896,
    "202410": 5_257_627,
    "202411": 5_883_102,
    "202412": 6_511_640,
    "202501": 5_552_541,
    "202502": 5_252_701,
    "202503": 6_208_631,
    "202504": 6_278_725,
    "202505": 6_608_720,
    "202506": 6_607_567,
    "202507": 6_533_487,
    "202508": 7_011_211,
    "202509": 7_500_365,
    "202510": 6_501_821,
    "202511": 6_829_087,
    "202512": 7_200_640,
}
INPUTS = {
    **{f"raw_{month}": path for month, path in RAW_INPUTS.items()},
    # 2026-11-08 version (commit 0d658ca): 109 stations, all with StationID (adds 臺大人文館 and 羅斯福路三段283巷).
    "stations_in_field": "data/processed/stations_in_field.csv",
}
EXPECTED_INPUT_SHA256 = {
    **{f"raw_{month}": EXPECTED_RAW_SHA256[month] for month in MONTHS},
    "stations_in_field": "74dadd7afe58f0b69b0a511f234dc85388e64290d5db420b8426767bad637f9e",
}
OUT_DIR = "data/processed/youbike_traffic"
OUTPUTS = {
    "traffic_2024": f"{OUT_DIR}/YouBike_traffic_2024_v7.csv",
    "traffic_2025": f"{OUT_DIR}/YouBike_traffic_2025_v7.csv",
    "traffic_example": f"{OUT_DIR}/YouBike_traffic_processed_example_v7.csv",
}
# All outputs go to data/processed/youbike_traffic/. The two full files exceed GitHub's
# 100 MB limit and are listed in that folder's .gitignore; the small example can be committed.
OUTPUT_AREAS = {"traffic_2024": "processed", "traffic_2025": "processed", "traffic_example": "processed"}
YEAR_OUTPUT = {2024: "traffic_2024", 2025: "traffic_2025"}

# Text reads: empty strings stay "" (not NaN) so blank bike_type / station names are preserved.
TEXT_READ = {"dtype": str, "keep_default_na": False, "na_filter": False, "encoding": "utf-8-sig"}
CHUNK_ROWS = 500_000
TRAFFIC_COLUMNS = ["rent_time", "rent_station", "return_time", "return_station", "rent", "bike_type", "infodate"]
OUTPUT_COLUMNS = ["rent_time", "rent_station", "rent_station_id", "return_time", "return_station",
                  "return_station_id", "rent", "bike_type", "infodate", "trip_group"]

input_paths = {}
for key, relative_path in INPUTS.items():
    area = Path(relative_path).parts[1]
    if area not in {"raw", "processed"}:
        raise ValueError(f"Expected a raw or upstream processed input: {relative_path}")
    input_paths[key] = project_file(relative_path, area)
output_paths = {key: project_file(path, OUTPUT_AREAS[key], must_exist=False) for key, path in OUTPUTS.items()}
if len(set(output_paths.values())) != len(output_paths):
    raise ValueError("Each output needs a distinct file path.")
if set(input_paths.values()) & set(output_paths.values()):
    raise ValueError("An output must not overwrite one of its inputs.")
existing = [str(path) for path in output_paths.values() if path.exists()]
if existing:
    raise FileExistsError(f"Review existing outputs before replacing them: {existing}")

if set(EXPECTED_INPUT_SHA256) != set(INPUTS):
    raise ValueError("Every input must have its own expected SHA-256.")
input_fingerprints = {key: verify_input_sha256(path, EXPECTED_INPUT_SHA256[key])
                      for key, path in input_paths.items()}
print(f"{len(input_fingerprints)} inputs match their expected SHA-256 "
      f"(24 raw months, stations_in_field).")

# Reuse the verified stage-1/stage-2 readers: binary line tracking, surrogateescape for invalid
# UTF-8, quoted fields, per-row column-count checks, and 6-column months padded with bike_type.
traffic_code_dir = project_file(f"{BASE_DIR}/YouBike_traffic_raw.py", "raw").parent
sys.path.insert(0, str(traffic_code_dir))
import YouBike_traffic_raw as stage2  # noqa: E402  (also imports station_dataprocessing as stage1)
assert stage2.COLUMNS == TRAFFIC_COLUMNS

stations = pd.read_csv(input_paths["stations_in_field"], **TEXT_READ)
print({"input": "stations_in_field", "rows": len(stations), "columns": list(stations.columns)})

25 inputs match their expected SHA-256 (24 raw months, stations_in_field).
{'input': 'stations_in_field', 'rows': 113, 'columns': ['StationUID', 'StationID', 'StationNameZh_tw', 'StationNameEn', 'PositionLat', 'PositionLon', 'first_seen', 'last_seen', 'days_observed']}


## 站名 → StationID 對照（109 站）

In [3]:
# Station-name -> StationID lookup: the 109 stations of stations_in_field (2026-11-08 version).
PREFIX = "YouBike2.0_"
OUTSIDE_ID = "-1"   # not in the study field
TOKEN_PREFIX = "<UNRESOLVED_UTF8_HEX:"

assert len(stations) == 113 and stations["StationUID"].nunique() == 109
assert (stations != "").all().all(), "stations_in_field has blank cells."
assert stations["StationNameZh_tw"].str.startswith(PREFIX).all()
assert stations["StationID"].str.fullmatch(r"\d{9}").all()

name_to_id = stations[["StationNameZh_tw", "StationID"]].drop_duplicates()
assert name_to_id["StationNameZh_tw"].is_unique and name_to_id["StationID"].is_unique
assert len(name_to_id) == 109
STATION_ID = dict(zip(name_to_id["StationNameZh_tw"], name_to_id["StationID"]))

# Stations added in the 2026-11-08 station list; both start in 2025 (no 2024 snapshot).
# 臺大人文館 used a placeholder ID (-2) before this version; it now has its real StationID.
NEW_2025_STATIONS = {"臺大人文館": "500119097", "羅斯福路三段283巷": "500101265"}
for name, station_id in NEW_2025_STATIONS.items():
    assert STATION_ID[PREFIX + name] == station_id, name

TARGETS = {name.removeprefix(PREFIX) for name in STATION_ID}
for name in TARGETS:
    assert name == name.strip() and name and not stage2.stage1.suspicious(name), name
# Raw names that are truncated forms of an in-field station name (checked against all 24 raw months, 2026-11-08):
# 2024-01..2025-01 write 捷運科技大樓站(台北教育大學) (StationID 500101166) without the closing parenthesis.
# They are mapped to the full name before filtering, so these trips get the correct name and StationID.
NAME_ALIASES = {"捷運科技大樓站(台北教育大學": "捷運科技大樓站(台北教育大學)"}
for raw_name, full_name in NAME_ALIASES.items():
    assert raw_name not in TARGETS and full_name in TARGETS, (raw_name, full_name)
print({"stations with real ID": len(STATION_ID), "added in 2025": NEW_2025_STATIONS, "filter names": len(TARGETS),
       "name aliases": NAME_ALIASES})


def add_prefix(names):
    """Prefix every non-empty station name; keep blanks and unresolved-byte tokens unchanged."""
    keep = (names == "") | names.str.startswith(TOKEN_PREFIX) | names.str.startswith(PREFIX)
    return names.where(keep, PREFIX + names)


# trip_group (team definition, 2026-11-08); mutually exclusive, d is checked first:
#   a = 外借台大還（流入）: rent outside (-1), return inside
#   b = 台大借外還（流出）: rent inside, return outside (-1)
#   c = 台大借台大還（區內）: both inside, not d
#   d = 無效還車: same station and ride time (rent) <= 180 seconds
# "Inside" = any ID other than -1 (the 109 stations).
TRIP_GROUPS = {"a": "外借台大還（流入）", "b": "台大借外還（流出）", "c": "台大借台大還（區內，不含 d）",
               "d": "無效還車（同站借還且時長 ≤ 3 分鐘）"}
INVALID_MAX_SECONDS = 180


def duration_seconds(rent):
    """rent is 'H:MM:SS' (hours may exceed two digits)."""
    return (rent.str[:-6].astype(np.int64) * 3600 + rent.str[-5:-3].astype(np.int64) * 60
            + rent.str[-2:].astype(np.int64))


def trip_group(frame):
    rent_in = frame["rent_station_id"] != OUTSIDE_ID
    return_in = frame["return_station_id"] != OUTSIDE_ID
    short_same = (frame["rent_station"] == frame["return_station"]) & (duration_seconds(frame["rent"]) <= INVALID_MAX_SECONDS)
    group = np.select([short_same, ~rent_in & return_in, rent_in & ~return_in, rent_in & return_in],
                      ["d", "a", "b", "c"], default="?")
    assert (group != "?").all(), "A row has neither end in the field."
    return group


def transform(chunk):
    out = chunk.copy()
    for side in ("rent", "return"):
        out[f"{side}_station"] = add_prefix(chunk[f"{side}_station"])
        # -1 = outside the field (or blank / unreadable name).
        out[f"{side}_station_id"] = out[f"{side}_station"].map(STATION_ID).fillna(OUTSIDE_ID)
    out["trip_group"] = trip_group(out)
    return out[OUTPUT_COLUMNS]


demo = pd.DataFrame({c: ["x"] * 6 for c in TRAFFIC_COLUMNS})
demo["rent_station"] = ["臺大人文館", "", "臺大禮賢樓東南側", "臺大禮賢樓東南側", "臺大禮賢樓東南側", "羅斯福路三段283巷"]
demo["return_station"] = ["立人國小", "臺大禮賢樓東南側", "臺大水源舍區A棟", "臺大禮賢樓東南側", "臺大禮賢樓東南側", "臺大人文館"]
demo["rent"] = ["00:10:00", "00:10:00", "00:05:00", "00:03:00", "00:03:01", "00:10:00"]
demo_out = transform(demo)
print(demo_out[["rent_station", "rent_station_id", "return_station", "return_station_id", "rent", "trip_group"]].to_string(index=False))
assert demo_out["rent_station_id"].tolist() == ["500119097", "-1", "500119044", "500119044", "500119044", "500101265"]
assert demo_out["return_station_id"].tolist() == ["-1", "500119044", "500119005", "500119044", "500119044", "500119097"]
assert demo_out["trip_group"].tolist() == ["b", "a", "c", "d", "c", "c"]

{'stations with real ID': 109, 'added in 2025': {'臺大人文館': '500119097', '羅斯福路三段283巷': '500101265'}, 'filter names': 109, 'name aliases': {'捷運科技大樓站(台北教育大學': '捷運科技大樓站(台北教育大學)'}}
         rent_station rent_station_id      return_station return_station_id     rent trip_group
     YouBike2.0_臺大人文館       500119097     YouBike2.0_立人國小                -1 00:10:00          b
                                   -1 YouBike2.0_臺大禮賢樓東南側         500119044 00:10:00          a
  YouBike2.0_臺大禮賢樓東南側       500119044 YouBike2.0_臺大水源舍區A棟         500119005 00:05:00          c
  YouBike2.0_臺大禮賢樓東南側       500119044 YouBike2.0_臺大禮賢樓東南側         500119044 00:03:00          d
  YouBike2.0_臺大禮賢樓東南側       500119044 YouBike2.0_臺大禮賢樓東南側         500119044 00:03:01          c
YouBike2.0_羅斯福路三段283巷       500101265    YouBike2.0_臺大人文館         500119097 00:10:00          c


## 從 24 個原始月檔重新篩選並寫出 v7

逐月讀取（沿用 stage2 讀取函式），保留借車或還車任一端在 109 個站名中的交易，同時加前綴與 StationID。每月讀到的 bytes 與筆數都要和第一階段稽核一致。

In [4]:
# Re-filter every raw month: keep a trip when either end exactly matches one of the 109 names (OR).
# Year follows the source-file month, as in stage 2. No dedup or imputation; the only renaming is NAME_ALIASES,
# applied to both ends before filtering.
import time

monthly = []
annual_rows = Counter()
with output_paths["traffic_2024"].open("x", encoding="utf-8-sig", newline="") as out_2024, \
     output_paths["traffic_2025"].open("x", encoding="utf-8-sig", newline="") as out_2025:
    streams = {2024: out_2024, 2025: out_2025}
    for stream in streams.values():
        stream.write(",".join(OUTPUT_COLUMNS) + "\n")
    for month in MONTHS:
        started = time.time()
        year = int(month[:4])
        state = Counter()  # stage2.chunks fills original_rows, source_sha256, csv_read_complete
        stats = Counter()
        for frame in stage2.chunks(input_paths[f"raw_{month}"], month, state):
            for side in ("rent_station", "return_station"):
                aliased = frame[side].isin(list(NAME_ALIASES))
                stats[f"{side.split('_')[0]}_aliased"] += int(aliased.sum())
                frame.loc[aliased, side] = frame.loc[aliased, side].map(NAME_ALIASES)
            rent_hit = frame["rent_station"].isin(TARGETS)
            return_hit = frame["return_station"].isin(TARGETS)
            keep = rent_hit | return_hit
            stats["rent_matched"] += int(rent_hit.sum())
            stats["return_matched"] += int(return_hit.sum())
            stats["both_matched"] += int((rent_hit & return_hit).sum())
            stats["kept"] += int(keep.sum())
            selected = stage2.serialize_selected(frame.loc[keep, TRAFFIC_COLUMNS].copy())
            selected = selected.fillna("").astype(str)  # bike_type is NA for 202401-202410
            transform(selected).to_csv(streams[year], index=False, header=False, lineterminator="\n")
        assert state["csv_read_complete"], f"{month}: read incomplete."
        assert state["source_sha256"] == EXPECTED_RAW_SHA256[month], f"{month}: bytes read differ from audit."
        assert state["original_rows"] == EXPECTED_RAW_ROWS[month], f"{month}: row count differs from audit."
        assert stats["kept"] == stats["rent_matched"] + stats["return_matched"] - stats["both_matched"]
        annual_rows[year] += stats["kept"]
        monthly.append({"month": month, "original_rows": state["original_rows"], **stats,
                        "seconds": round(time.time() - started)})

monthly = pd.DataFrame(monthly)
monthly["retention_rate"] = (monthly["kept"] / monthly["original_rows"]).round(4)
pd.set_option("display.width", 200)
print(monthly.to_string(index=False))
annual = monthly.assign(year=monthly["month"].str[:4]).groupby("year")[["original_rows", "kept"]].sum()
print(annual)
assert monthly["original_rows"].sum() == 133_966_680

 month  original_rows  rent_aliased  return_aliased  rent_matched  return_matched  both_matched    kept  seconds  retention_rate
202401        2746557          3841            3425        233882          225091        143530  315443        8          0.1149
202402        2590755          3583            3289        235395          226845        147856  314384        8          0.1213
202403        3472416          5722            5557        412203          401736        279914  534025       11          0.1538
202404        3798550          6174            5879        427596          414854        288367  554083       12          0.1459
202405        4338453          7795            7380        501717          489773        343763  647727       14          0.1493
202406        3773353          6649            6459        371078          361411        247229  485260       12          0.1286
202407        5528656          8645            8122        510468          497196        346662  

## 範例檔

In [5]:
# Example file: first 100 rows of the saved 2025 v7 file, in file order.
example_frame = pd.read_csv(output_paths["traffic_2025"], nrows=100, **TEXT_READ)
assert example_frame.shape == (100, 10) and list(example_frame.columns) == OUTPUT_COLUMNS
with output_paths["traffic_example"].open("x", encoding="utf-8-sig", newline="") as stream:
    example_frame.to_csv(stream, index=False, lineterminator="\n")
pd.set_option("display.max_columns", None)
display(example_frame.head(10))

,rent_time,rent_station,rent_station_id,return_time,return_station,return_station_id,rent,bike_type,infodate,trip_group
0,2025-01-13 15:00:00,YouBike2.0_捷運古亭站(8號出口),-1,2025-01-13 21:00:00,YouBike2.0_新生南路三段52號前,500101008,05:34:25,電輔車,2025-01-13,a
1,2025-01-13 01:00:00,YouBike2.0_辛亥路一段30號前,500101012,2025-01-13 03:00:00,YouBike2.0_捷運公館站(4號出口),500106004,01:50:15,一般車,2025-01-13,c
2,2025-01-13 11:00:00,YouBike2.0_臺大土木研究大樓前,500101035,2025-01-13 12:00:00,YouBike2.0_立人國小,-1,00:15:08,一般車,2025-01-13,b
3,2025-01-13 12:00:00,YouBike2.0_臺大男一舍前,500101032,2025-01-13 12:00:00,YouBike2.0_捷運公館站(2號出口),500101022,00:06:22,一般車,2025-01-13,c
4,2025-01-13 17:00:00,YouBike2.0_和平泰順街口,-1,2025-01-13 17:00:00,YouBike2.0_羅斯福路三段245號前,500101016,00:04:40,電輔車,2025-01-13,a
5,2025-01-13 15:00:00,YouBike2.0_臺大卓越研究大樓,500119006,2025-01-13 16:00:00,YouBike2.0_臺大立體機車停車場,500119051,00:35:58,一般車,2025-01-13,c
6,2025-01-13 16:00:00,YouBike2.0_臺大土木研究大樓前,500101035,2025-01-13 17:00:00,YouBike2.0_捷運萬隆站(2號出口),-1,00:52:45,一般車,2025-01-13,b
7,2025-01-13 15:00:00,YouBike2.0_捷運公館站(2號出口),500101022,2025-01-13 16:00:00,YouBike2.0_臺大鹿鳴堂東側,500119061,00:32:42,一般車,2025-01-13,c
8,2025-01-13 15:00:00,YouBike2.0_臺大小福樓東側,500119050,2025-01-13 16:00:00,YouBike2.0_臺大小福樓東側,500119050,00:38:56,一般車,2025-01-13,c
9,2025-01-13 13:00:00,YouBike2.0_臺大工綜館南側,500119052,2025-01-13 13:00:00,YouBike2.0_臺大二號館,500119065,00:03:08,一般車,2025-01-13,c


## 讀回驗證

In [6]:
# Reopen every saved output and check it.
listed = set(STATION_ID)
new_prefixed = [PREFIX + name for name in NEW_2025_STATIONS]
# 臺大人文館 and 羅斯福路三段283巷 first appear in 2025, so 2024 has 107 stations with trips.
EXPECTED_STATIONS_SEEN = {2024: 107, 2025: 109}
verification = {}
group_counts = {}
d_months = Counter()
for year, out_key in YEAR_OUTPUT.items():
    rows, token_cells, blank_cells = 0, 0, 0
    new_rows = Counter()
    id_seen = set()
    groups = Counter()
    for saved in pd.read_csv(output_paths[out_key], chunksize=CHUNK_ROWS, **TEXT_READ):
        assert list(saved.columns) == OUTPUT_COLUMNS, f"Schema changed while saving {out_key}."
        rent_in, return_in = saved["rent_station"].isin(listed), saved["return_station"].isin(listed)
        assert (rent_in | return_in).all(), f"{out_key}: a row has neither end in the station list."
        for side, in_list in (("rent", rent_in), ("return", return_in)):
            names, ids = saved[f"{side}_station"], saved[f"{side}_station_id"]
            is_token = names.str.startswith(TOKEN_PREFIX)
            assert not names.isin([PREFIX + n for n in NAME_ALIASES]).any(), "A truncated alias name was left in the output."
            assert (names[(names != "") & ~is_token].str.startswith(PREFIX)).all()
            assert (ids[in_list] == names[in_list].map(STATION_ID)).all()
            assert (ids[~in_list] == OUTSIDE_ID).all()
            assert ids.str.fullmatch(r"\d{9}|-1").all(), "IDs must be 9-digit StationIDs or -1."
            id_seen.update(ids[in_list])
            token_cells += int(is_token.sum())
            blank_cells += int((names == "").sum())
        assert (saved["trip_group"].to_numpy() == trip_group(saved)).all(), f"{out_key}: trip_group differs from its rule."
        groups.update(saved["trip_group"].value_counts().to_dict())
        d_months.update(saved.loc[saved["trip_group"] == "d", "infodate"].str[:7].value_counts().to_dict())
        for name in new_prefixed:
            new_rows[name.removeprefix(PREFIX)] += int(((saved["rent_station"] == name) | (saved["return_station"] == name)).sum())
        rows += len(saved)
    assert rows == annual_rows[year], f"{out_key}: row count changed while saving."
    assert len(id_seen) == EXPECTED_STATIONS_SEEN[year], f"{out_key}: {len(id_seen)} stations appear."
    assert sum(groups.values()) == rows
    group_counts[year] = groups
    verification[out_key] = {"data_path": OUTPUTS[out_key], "rows": rows, "columns": len(OUTPUT_COLUMNS),
                             "stations appearing": len(id_seen),
                             "rows touching 2025 new stations": {name: new_rows[name] for name in NEW_2025_STATIONS},
                             "unresolved-token cells": token_cells, "blank-name cells": blank_cells,
                             "sha256": sha256_file(output_paths[out_key]), "status": "read-back checks passed"}
    print(verification[out_key])

example_saved = pd.read_csv(output_paths["traffic_example"], **TEXT_READ)
pd.testing.assert_frame_equal(example_saved, pd.read_csv(output_paths["traffic_2025"], nrows=100, **TEXT_READ))
verification["traffic_example"] = {"data_path": OUTPUTS["traffic_example"], "rows": len(example_saved),
                                   "columns": example_saved.shape[1],
                                   "sha256": sha256_file(output_paths["traffic_example"]),
                                   "status": "read-back checks passed (= first 100 rows of 2025 v7)"}
print(verification["traffic_example"])

group_table = pd.DataFrame({y: {g: group_counts[y].get(g, 0) for g in TRIP_GROUPS} for y in group_counts})
group_table["合計"] = group_table.sum(axis=1)
group_table["占全部"] = (group_table["合計"] / group_table["合計"].sum()).map(lambda x: f"{x:.2%}")
group_table.insert(0, "說明", pd.Series(TRIP_GROUPS))
for col in (2024, 2025, "合計"):
    group_table[col] = group_table[col].map(lambda x: f"{x:,}")
print("trip_group 各組列數")
display(group_table)
assert min(d_months) >= "2025-08", "d appears before 2025-08"
print("d 組各月列數（2025-08 以前資料來源沒有 5 分鐘內的同站還車）：", dict(sorted(d_months.items())))
assert all(sha256_file(path) == input_fingerprints[key] for key, path in input_paths.items()), "An input changed."
print("All inputs unchanged after processing.")

{'data_path': 'data/processed/youbike_traffic/YouBike_traffic_2024_v7.csv', 'rows': 7776981, 'columns': 10, 'stations appearing': 107, 'rows touching 2025 new stations': {'臺大人文館': 0, '羅斯福路三段283巷': 0}, 'unresolved-token cells': 820, 'blank-name cells': 1, 'sha256': '07fed70cbf742a933970c21fc7fbbc5a77b56b9e76c356fe0226c3bf4255757c', 'status': 'read-back checks passed'}
{'data_path': 'data/processed/youbike_traffic/YouBike_traffic_2025_v7.csv', 'rows': 10253912, 'columns': 10, 'stations appearing': 109, 'rows touching 2025 new stations': {'臺大人文館': 99221, '羅斯福路三段283巷': 13337}, 'unresolved-token cells': 108, 'blank-name cells': 3, 'sha256': '206ebf307c9e73d61423069b8f66c3b6e64d5340f17778cfe00f76d42cd20f32', 'status': 'read-back checks passed'}
{'data_path': 'data/processed/youbike_traffic/YouBike_traffic_processed_example_v7.csv', 'rows': 100, 'columns': 10, 'sha256': '45a38d74e7fec3d6ff2952388476e2d27943d08137396905f767c3e85f85bf63', 'status': 'read-back checks passed (= first 100 rows of 

,說明,2024,2025,合計,占全部
a,外借台大還（流入）,"1,710,313","2,210,188","3,920,501",21.74%
b,台大借外還（流出）,"1,858,538","2,403,320","4,261,858",23.64%
c,台大借台大還（區內，不含 d）,"4,208,130","5,637,675","9,845,805",54.61%
d,無效還車（同站借還且時長 ≤ 3 分鐘）,0,"2,729","2,729",0.02%


## 結果摘要

以下依 2026-11-08 在全新的 in-process IPython shell 由上而下完整執行的實際輸出整理（本機沒有 Jupyter）。本版（v7）比 v6 多了 `NAME_ALIASES` 的站名修正，並改用 2026-11-08 版 `stations_in_field`（109 站），其餘規則與 v6 相同。

- **輸入**：25 個輸入檔的 SHA-256 都符合預期；24 個月逐月讀到的 bytes 和筆數都與第一階段稽核一致，合計 133,966,680 筆。處理後所有輸入檔都沒有變動。
- **截斷站名修正**：`捷運科技大樓站(台北教育大學` 只出現在 2024-01～2025-01，2025-02 起原始資料全部是完整站名。改名的格數如下：
  - 2024：借車端 88,092 格、還車端 83,155 格
  - 2025（只有 1 月）：借車端 7,108 格、還車端 6,742 格
- **輸出與驗證**：三個檔案都已重新讀回，所有 assertion 通過，輸出中已沒有截斷站名，也沒有 `-2`。
  - `YouBike_traffic_2024_v7.csv`：7,776,981 列 × 10 欄（v6 為 7,659,827，多 117,154 列）；兩個 2025 新站都是 0 筆。
  - `YouBike_traffic_2025_v7.csv`：10,253,912 列 × 10 欄；99,221 列的一端是臺大人文館（`500119097`），13,337 列的一端是羅斯福路三段283巷（`500101265`）。
  - `YouBike_traffic_processed_example_v7.csv`：100 列 × 10 欄，與 2025 v7 前 100 列逐格相同。
  - 2024 年 107 站有交易，2025 年 109 站都有交易；`trip_group` 每列都符合分組規則。
- **改用 109 站清單前後比對**（2026-11-08 在本 notebook 之外另以程式比對，前版 = 臺大人文館用 `-2`、283巷算範圍外）：
  - 2024 年檔案與前版逐位元組相同（SHA-256 不變），因為兩個新站 2024 年都沒有交易。
  - 2025 年：兩端都不是 283巷的 10,240,575 列，把前版的 `-2` 換成 `500119097` 後逐列、逐格、順序完全相同。
  - 一端是 283巷的列從 5,656 增加到 13,337（多 7,681 列，都是 283巷 與範圍外站之間的旅次，前版兩端都不在範圍內而沒有收錄）。前版這 5,656 列分在 a（2,855）、b（2,801）；本版 13,337 列為 c 6,282、b 3,649、a 3,381、d 25。283巷 第一筆交易在 2025-10-10。
- **與 v6 比對**（2026-11-08 在本 notebook 之外另以程式比對；比對對象是改用 109 站清單前的 v7，2025 年列數因此不含 283巷 新增的列）：
  - 兩端都不是 `500101166` 的列，v7 與 v6 逐列、逐格、順序完全相同（2024：7,608,743 列；2025：10,038,545 列）。
  - `500101166` 的格數等於 v6 原有的完整站名格數加上改名格數：2024 借車 24 → 88,116、還車 27 → 83,182；2025 借車 102,970 → 110,078、還車 94,385 → 101,127。
  - 新增的列都是「一端是本站、另一端在範圍外」的交易，v6 因為兩端都不在清單內而整筆漏掉。v6 中原本被歸為 a／b 的列（截斷站名當成範圍外），在 v7 有一部分改歸 c。
  - 本站 2024 年的第一筆交易在 2024-01-01，不是 v6 顯示的 2024-07-07。
- **`trip_group` 各組列數**（兩年合計 18,030,893 列）：
  - a 外借台大還（流入）：3,920,501 列（21.74%）
  - b 台大借外還（流出）：4,261,858 列（23.64%）
  - c 台大借台大還（區內）：9,845,805 列（54.61%）
  - d 無效還車：2,729 列（0.02%），全部在 2025-08 以後
- **d 組只在 2025-08 以後出現**：2025-07 以前，資料來源就沒有 5 分鐘內的同站還車紀錄（同站借還最短幾乎都是 300–301 秒）；2025-08 起才出現 1–6 秒的紀錄（各月：2025-08 有 126 列，之後每月 561–724 列）。所以 d 組不能做跨期比較；2025-08 以後的 c 組也多了 3–5 分鐘的同站借還。
- **更早版本的比對**：v6 與加欄前 v6、分組檔及 v5 的比對結果見 git 歷史中的 `youbike_traffic_station_id_v6.ipynb`。
- **範例檔表格的顯示**：本次直接以 `display(...)` 顯示。
- **仍保留的限制**：
  - 臺大人文館（2025-02-04 起）與羅斯福路三段283巷（2025-10-10 起）只有 2025 年有交易，跨年比較需要控制站點數。
  - 原始資料另有還車站名 `和平公園`（2024 年 296 筆、2025 年 348 筆，沒有借車），與範圍內的 `和平公園東側` 名稱不同；是否為同一站尚未確認，未納入 `NAME_ALIASES`。
  - 亂碼 token 保持原樣：2024 年 820 格、2025 年 108 格；空白站名：2024 年 1 格、2025 年 3 格。

